# Migration des Données Médicales vers MongoDB

## Objectif
Ce notebook a pour but de démontrer les étapes nécessaires pour migrer un fichier de données médicales au format CSV vers une base de données MongoDB, avec les étapes suivantes :
1. Chargement et validation des données.
2. Nettoyage et export des données.
3. Insertion des données dans MongoDB.
4. Vérification des documents insérés dans la base de données.

## Fichiers requis
- **healthcare_dataset.csv** : Fichier source contenant les données médicales. Ce fichier doit être placé dans le répertoire `data/raw/`.

## Résultat attendu
Les données nettoyées seront enregistrées dans `data/processed/healthcare_data_cleaned.csv` et insérées dans une base MongoDB locale.


In [21]:
# Importation des bibliothèques nécessaires
import pandas as pd  # Pour la manipulation des données
import os  # Pour interagir avec le système de fichiers
from loguru import logger  # Pour gérer les logs
from pymongo import MongoClient  # Pour se connecter à MongoDB
from pathlib import Path  # Pour gérer les chemins de fichiers et répertoires
import sys  # Pour modifier le chemin d'import des modules
import kagglehub  # Pour télécharger les datasets Kaggle
from skimpy import skim  # Résumés statistiques enrichis pour les DataFrames
import warnings   # Suppression des avertissements inutiles

# --- Configuration des Affichages pandas ---
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes d'un DataFrame
pd.set_option('display.max_rows', 100)      # Affiche jusqu'à 100 lignes dans un DataFrame
pd.set_option('display.width', 400)         # Évite les retours à la ligne inutiles

# --- Suppression des Avertissements Inutiles ---
warnings.filterwarnings("ignore")

# Configuration des logs
# Les logs sont enregistrés dans un fichier avec une rotation automatique pour limiter la taille
logger.add("../logs/migration.log", rotation="500 MB")
logger.info("Démarrage du notebook")

# Définir le répertoire de base pour un notebook ou un script
# Si on est dans un notebook, on remonte au répertoire parent
NOTEBOOK_PATH = Path(os.getcwd())
if NOTEBOOK_PATH.name == "notebooks":
    BASE_DIR = NOTEBOOK_PATH.parent
else:
    BASE_DIR = NOTEBOOK_PATH

# Ajouter le chemin du répertoire 'src' au sys.path
sys.path.append(str(BASE_DIR / "src"))

# Définir les chemins des répertoires à partir du répertoire de base
RAW_DIR = BASE_DIR / "data/raw"  # Répertoire pour les fichiers bruts
PROCESSED_DIR = BASE_DIR / "data/processed"  # Répertoire pour les fichiers nettoyés

# Logs pour vérifier les répertoires définis
logger.info(f"Répertoire racine : {BASE_DIR}")
logger.info(f"Répertoire pour les fichiers bruts : {RAW_DIR}")
logger.info(f"Répertoire pour les fichiers nettoyés : {PROCESSED_DIR}")

# Création des répertoires si inexistants
# Cette étape garantit que tous les répertoires nécessaires sont disponibles avant de poursuivre
try:
    # Vérifie ou crée le répertoire pour les fichiers bruts
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    logger.info(f"Répertoire 'RAW_DIR' vérifié ou créé : {RAW_DIR}")

    # Vérifie ou crée le répertoire pour les fichiers nettoyés
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    logger.info(f"Répertoire 'PROCESSED_DIR' vérifié ou créé : {PROCESSED_DIR}")
except Exception as e:
    # Log d'une erreur si un répertoire ne peut pas être créé
    logger.error(f"Erreur lors de la création des répertoires : {e}")
    raise  # Arrête l'exécution en cas d'erreur critique

logger.info("Répertoires vérifiés et créés avec succès.")


2024-12-31 17:52:39.141 | INFO     | __main__:<module>:23 - Démarrage du notebook
2024-12-31 17:52:39.145 | INFO     | __main__:<module>:41 - Répertoire racine : f:\1.Boulot\02_Openclassrooms\Github\migration-donnees-mongodb
2024-12-31 17:52:39.147 | INFO     | __main__:<module>:42 - Répertoire pour les fichiers bruts : f:\1.Boulot\02_Openclassrooms\Github\migration-donnees-mongodb\data\raw
2024-12-31 17:52:39.148 | INFO     | __main__:<module>:43 - Répertoire pour les fichiers nettoyés : f:\1.Boulot\02_Openclassrooms\Github\migration-donnees-mongodb\data\processed
2024-12-31 17:52:39.150 | INFO     | __main__:<module>:50 - Répertoire 'RAW_DIR' vérifié ou créé : f:\1.Boulot\02_Openclassrooms\Github\migration-donnees-mongodb\data\raw
2024-12-31 17:52:39.152 | INFO     | __main__:<module>:54 - Répertoire 'PROCESSED_DIR' vérifié ou créé : f:\1.Boulot\02_Openclassrooms\Github\migration-donnees-mongodb\data\processed
2024-12-31 17:52:39.154 | INFO     | __main__:<module>:60 - Répertoires vé

In [22]:
# -------------------------------------------
# Téléchargement et Chargement des Données
# -------------------------------------------

# Téléchargement du dataset depuis Kaggle
try:
    # Télécharge le dataset en utilisant KaggleHub
    dataset_path = kagglehub.dataset_download("prasad22/healthcare-dataset")
    logger.info(f"Dataset téléchargé avec succès dans : {dataset_path}")
except Exception as e:
    # Gère les erreurs de téléchargement et les logue
    logger.error(f"Erreur lors du téléchargement du dataset : {e}")
    raise  # Arrête l'exécution en cas d'échec

# Étape 2 : Lister les fichiers disponibles dans le dossier téléchargé
try:
    # Liste tous les fichiers dans le répertoire du dataset
    files = os.listdir(dataset_path)
    logger.info(f"Fichiers disponibles dans le dataset : {files}")
except Exception as e:
    # Log en cas d'échec de la lecture du répertoire
    logger.error(f"Erreur lors de la lecture des fichiers dans le répertoire {dataset_path} : {e}")
    raise

# Étape 3 : Chargement d'un fichier spécifique
# Nom du fichier attendu
dataset_file = Path(dataset_path) / "healthcare_dataset.csv"
try:
    # Charge le fichier CSV en mémoire en tant que DataFrame pandas
    data = pd.read_csv(dataset_file)
    logger.info(f"Fichier '{dataset_file.name}' chargé avec succès. "
                f"Nombre de lignes : {data.shape[0]}. Nombre de colonnes : {data.shape[1]}.")
except FileNotFoundError:
    # Log en cas de fichier manquant
    logger.error(f"Le fichier spécifié '{dataset_file.name}' est introuvable dans le dataset.")
    raise
except Exception as e:
    # Log pour tout autre type d'erreur
    logger.error(f"Erreur lors du chargement du fichier '{dataset_file.name}' : {e}")
    raise


2024-12-31 17:52:42.759 | INFO     | __main__:<module>:9 - Dataset téléchargé avec succès dans : C:\Users\rouss\.cache\kagglehub\datasets\prasad22\healthcare-dataset\versions\2
2024-12-31 17:52:42.764 | INFO     | __main__:<module>:19 - Fichiers disponibles dans le dataset : ['healthcare_dataset.csv']
2024-12-31 17:52:42.877 | INFO     | __main__:<module>:31 - Fichier 'healthcare_dataset.csv' chargé avec succès. Nombre de lignes : 55500. Nombre de colonnes : 15.


In [23]:
# Aperçu des données
# Afficher les 5 premières lignes pour un aperçu rapide du contenu
logger.info(f"Aperçu des données : \n{data.head()}\n")

# Résumé statistique avec skim
# Fournit une vue d'ensemble rapide des colonnes, types, valeurs manquantes, etc.
logger.info(f"Résumé pour le dataset 'healthcare_dataset':")
skim(data)  # Assurez-vous que skimpy est installé : pip install skimpy

# Séparateur visuel dans les logs
logger.info(f"{'=' * 60}")

# Types des colonnes
# Affiche les types des colonnes pour identifier d'éventuels problèmes
logger.info(f"📊 Types des colonnes pour le DataFrame 'healthcare'")
logger.info(f"{'-' * 60}")

# Boucle pour afficher les détails de chaque colonne
# Cela inclut le type de données pandas et les types Python uniques rencontrés
for col in data.columns:
    col_type = data[col].dtype  # Type de colonne selon pandas (int64, object, etc.)
    unique_types = data[col].apply(lambda x: type(x).__name__).unique()  # Types Python rencontrés
    logger.info(f"🔹 {col} : {col_type} | Valeurs : {', '.join(unique_types)}")

# Séparateur final
logger.info(f"{'=' * 60}\n")


2024-12-31 17:52:45.736 | INFO     | __main__:<module>:3 - Aperçu des données : 
            Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                    Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31     Matthew Smith             Sons and Miller         Blue Cross    18856.281306          328         Urgent     2024-02-02  Paracetamol        Normal
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   Samantha Davies                     Kim Inc           Medicare    33643.327287          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell                    Cook PLC              Aetna    27955.096079          205      Emergency     2022-10-07      Aspirin        Normal
3   and

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 55500  │ │ string      │ 12    │                                                          │
│ │ Number of columns │ 15     │ │ int32       │ 2     │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column_name       ┃ NA  ┃ NA %   ┃ mean    ┃ sd     ┃ p0     ┃ p25    ┃ p50    ┃ p75    ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │ Age               │   0 │      0 │   51.54 │   19.6 │     13 │     35 │     52 │     68 │     89 │ ▅▇▇▇▇▅  │  │
│ │ Billing Amount    │   0 │      0 │   25540 │  14210 │  -2008 │  13240 │  25540 │  37820 │  52760 │ ▅▇▇▇▇▆  │  │
│ │ Room Number       │   0 │      0 │   301.1 │  115.2 │    101 │    202 │    302 │    401 │    500 │ ▇▇▇▇▇▇  │  │
│ └───────────────────┴─────┴────────┴─────────┴────────┴────────┴────────┴────────┴────────┴────────┴─────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column_name                        ┃ NA     ┃ NA %      ┃ words per row             ┃ total words          ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │ Name                               │      0 │         0 │                         2 │               113390 │  │
│ │ Gender                             │      0 │         0 │                         1 │                55500 │  │
│ │ Blood Type                         │      0 │         0 │                         1 │                55500 │  │
│ │ Medical Condition                  │      0 │         0 │                         1 │                55500 │  │
│ │ Date of Admission                  │      0 │         0 │                         1 │                55500 │  │
│ │ Doctor                             │      0 │         0 │                         2 │               113516 │  │
│ │ Hospital                           │      0 │         0 │                       2.4 │               133009 │  │
│ │ Insurance Provider                 │      0 │         0 │                       1.2 │                66559 │  │
│ │ Admission Type                     │      0 │         0 │                         1 │                55500 │  │
│ │ Discharge Date                     │      0 │         0 │                         1 │                55500 │  │
│ │ Medication                         │      0 │         0 │                         1 │                55500 │  │
│ │ Test Results                       │      0 │         0 │                         1 │                55500 │  │
│ └────────────────────────────────────┴────────┴───────────┴───────────────────────────┴──────────────────────┘  │
╰────────────────────────────────────────────────────── 

2024-12-31 17:52:46.079 | INFO     | __main__:<module>:11 - ============================================================
2024-12-31 17:52:46.081 | INFO     | __main__:<module>:15 - 📊 Types des colonnes pour le DataFrame 'healthcare'
2024-12-31 17:52:46.083 | INFO     | __main__:<module>:16 - ------------------------------------------------------------
2024-12-31 17:52:46.094 | INFO     | __main__:<module>:23 - 🔹 Name : object | Valeurs : str
2024-12-31 17:52:46.105 | INFO     | __main__:<module>:23 - 🔹 Age : int64 | Valeurs : int
2024-12-31 17:52:46.116 | INFO     | __main__:<module>:23 - 🔹 Gender : object | Valeurs : str
2024-12-31 17:52:46.127 | INFO     | __main__:<module>:23 - 🔹 Blood Type : object | Valeurs : str
2024-12-31 17:52:46.140 | INFO     | __main__:<module>:23 - 🔹 Medical Condition : object | Valeurs : str
2024-12-31 17:52:46.153 | INFO     | __main__:<module>:23 - 🔹 Date of Admission : object | Valeurs : str
2024-12-31 17:52:46.164 | INFO     | __main__:<module>:23 - 🔹 